In [2]:
import pandas as pd

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3864 entries, 0 to 3863
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              3864 non-null   datetime64[ns]
 1   agegroup          3864 non-null   object        
 2   Gender            3864 non-null   object        
 3   Health Zone       3864 non-null   object        
 4   Health Zone Name  3864 non-null   object        
 5   Chronic Disease   3864 non-null   object        
 6   prevalence        3864 non-null   int64         
 7   population        3864 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 241.6+ KB
None


,Date,agegroup,Gender,Health Zone,Health Zone Name,Chronic Disease,prevalence,population
0,2000-01-01,20 to 29,F,Zone 4,Central,Diabetes,223,30198
1,2000-01-01,30 to 39,F,Zone 4,Central,Diabetes,455,34639
2,2000-01-01,40 to 49,F,Zone 4,Central,Diabetes,934,32980
3,2000-01-01,50 to 59,F,Zone 4,Central,Diabetes,1667,24173
4,2000-01-01,60 to 69,F,Zone 4,Central,Diabetes,1818,14990


In [7]:
import pandas as pd

# ---------------------------------------------------
# Helper: Split Health Zone Code & Name
# ---------------------------------------------------
def split_zone(zone_str):
    """
    'Zone 4 - Central' -> ('Zone 4', 'Central')
    """
    zone_code, zone_name = zone_str.split('-', 1)
    return zone_code.strip(), zone_name.strip()


# ---------------------------------------------------
# Generic Cleaner
# ---------------------------------------------------
def clean_chronic_file(file_path, disease_name, prevalence_column, force_gender=None):
    df = pd.read_csv(file_path)

    # Normalize column names
    df.columns = df.columns.str.lower().str.strip()

    # Standardize column names
    df = df.rename(columns={
        'sex': 'gender',
        'age_group': 'agegroup',
        'year': 'date',
        prevalence_column: 'prevalence'
    })

    # Date
    df['date'] = pd.to_datetime(df['date'])

    # Gender handling
    if force_gender is not None:
        df['gender'] = force_gender
    else:
        df['gender'] = df['gender'].replace({
            'M': 'M',
            'F': 'F',
            'All': 'All'
        })

    # Split zone fields
    df[['Health Zone', 'Health Zone Name']] = (
        df['zone'].apply(lambda x: pd.Series(split_zone(x)))
    )

    # Add disease label
    df['Chronic Disease'] = disease_name

    # Final schema
    df = df[[
        'date',
        'agegroup',
        'gender',
        'Health Zone',
        'Health Zone Name',
        'Chronic Disease',
        'prevalence',
        'population'
    ]]

    # Rename columns
    df = df.rename(columns={
        'date': 'Date',
        'gender': 'Gender'
    })

    return df


# ---------------------------------------------------
# Existing chronic diseases
# ---------------------------------------------------
diabetes_df = clean_chronic_file(
    './data/Diabetes Crude Prevalence.csv',
    'Diabetes',
    'prevalence'
)

hypertension_df = clean_chronic_file(
    './data/Hypertension Crude Prevalence.csv',
    'Hypertension',
    'hypertension_count'
)

ihd_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Ischemic Heart Disease',
    'prevalence'
)


# ---------------------------------------------------
# NEW: Acute Myocardial Infarction (AMI → Gender = All)
# ---------------------------------------------------
ami_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Acute Myocardial Infarction (AMI)',
    'prevalence',
    force_gender='All'
)

ami_df = ami_df[ami_df['Chronic Disease'] == 'Acute Myocardial Infarction (AMI)']


# ---------------------------------------------------
# NEW: Asthma (gender already exists)
# ---------------------------------------------------
asthma_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Asthma',
    'prevalence'
)

asthma_df = asthma_df[asthma_df['Chronic Disease'] == 'Asthma']


# ---------------------------------------------------
# Append into ONE dataset
# ---------------------------------------------------
chronic_diseases = pd.concat(
    [
        diabetes_df,
        hypertension_df,
        ihd_df,
        ami_df,
        asthma_df
    ],
    ignore_index=True
)


# ---------------------------------------------------
# Final validation
# ---------------------------------------------------
print(chronic_diseases['Chronic Disease'].value_counts())
display(chronic_diseases.head())


# ---------------------------------------------------
# Save output
# ---------------------------------------------------
chronic_diseases.to_csv(
    './clean/chronic_diseases.csv',
    index=False
)

Chronic Disease
Diabetes                             1288
Hypertension                         1288
Ischemic Heart Disease               1288
Acute Myocardial Infarction (AMI)    1288
Asthma                               1288
Name: count, dtype: int64


,Date,agegroup,Gender,Health Zone,Health Zone Name,Chronic Disease,prevalence,population
0,2000-01-01,20 to 29,F,Zone 4,Central,Diabetes,223,30198
1,2000-01-01,30 to 39,F,Zone 4,Central,Diabetes,455,34639
2,2000-01-01,40 to 49,F,Zone 4,Central,Diabetes,934,32980
3,2000-01-01,50 to 59,F,Zone 4,Central,Diabetes,1667,24173
4,2000-01-01,60 to 69,F,Zone 4,Central,Diabetes,1818,14990
